# Burn Area Detection — Analysis Notebook
## Dixie Fire 2021 · Sentinel-2 · Threshold + Random Forest

This notebook walks through the full pipeline step-by-step with visualisations at each stage.
Use this to understand the data before running `src/pipeline.py`.

**Prerequisites:** GeoTIFFs downloaded from Google Drive into `data/`

---
## 0. Setup

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from pathlib import Path

from src.indices import load_stack, get_band, classify_binary, classify_dnbr, burn_severity_summary, build_feature_matrix
from src.classifier import ThresholdClassifier, BurnRFClassifier, compare_methods
from src.visualise import (
    plot_dnbr_distribution, plot_severity_map,
    plot_method_comparison, plot_feature_importance, build_folium_map
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Setup complete')

/home/krithi/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Setup complete


---
## 1. Load the Analysis Stack

In [2]:
STACK_PATH   = '../data/processed/analysis_stack_dixie2021.tif'
SAMPLES_PATH = '../data/samples/training_samples_dixie2021.csv'

data, profile = load_stack(STACK_PATH)
print(f'Stack shape : {data.shape}  (bands, height, width)')
print(f'CRS         : {profile["crs"]}')
print(f'Resolution  : {profile["transform"].a:.1f}m')
print(f'Nodata      : {profile.get("nodata")}')

RasterioIOError: ../data/processed/analysis_stack_dixie2021.tif: No such file or directory

---
## 2. Explore dNBR Signal

In [ ]:
dNBR  = get_band(data, 'dNBR')
rdNBR = get_band(data, 'RdNBR')

print('dNBR statistics:')
flat = dNBR.ravel()
flat = flat[np.isfinite(flat)]
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f'  p{p:>3}: {np.percentile(flat, p):+.4f}')

# Sanity check: 90th percentile should be > 0.27 if burn signal is present
p90 = np.percentile(flat, 90)
print(f'\n90th percentile = {p90:.4f}')
print('✓ Strong burn signal' if p90 > 0.3 else '⚠ Weak signal — check date windows')

In [ ]:
fig = plot_dnbr_distribution(dNBR)
plt.show()

---
## 3. Threshold Classification

In [ ]:
thresh_clf    = ThresholdClassifier(threshold=0.27)
threshold_map = thresh_clf.predict(data)
severity_map  = classify_dnbr(dNBR)

print(f'Burned pixels   : {threshold_map.sum():,}')
print(f'Total pixels    : {threshold_map.size:,}')
print(f'Burn coverage   : {100*threshold_map.mean():.1f}%')

summary = burn_severity_summary(severity_map)
pd.DataFrame(summary).T

In [ ]:
fig = plot_severity_map(dNBR, title='Burn Severity — Threshold Classification (USGS dNBR)')
plt.show()

---
## 4. Random Forest Training

In [ ]:
samples = pd.read_csv(SAMPLES_PATH)
print(f'Samples: {len(samples):,}')
print(samples['burned_label'].value_counts().rename({0:'Unburned', 1:'Burned'}))
samples.head()

In [ ]:
exclude_cols  = {'burned_label', 'system:index', '.geo', 'longitude', 'latitude'}
feature_cols  = [c for c in samples.columns if c not in exclude_cols]
X = samples[feature_cols].values
y = samples['burned_label'].values

rf_clf = BurnRFClassifier(n_estimators=200, max_depth=20)
metrics = rf_clf.fit(X, y, feature_names=feature_cols)

In [ ]:
print('Cross-validation (5-fold):')
cv = rf_clf.cross_validate(X, y, n_splits=5)

---
## 5. RF Inference + Method Comparison

In [ ]:
X_full, feat_names = build_feature_matrix(data)
H, W = data.shape[1], data.shape[2]

y_proba   = rf_clf.predict_proba(X_full)
rf_map    = (y_proba >= 0.5).astype(np.uint8).reshape(H, W)
proba_map = y_proba.reshape(H, W).astype(np.float32)

print(f'RF burned pixels    : {rf_map.sum():,}')
print(f'Threshold burned px : {threshold_map.sum():,}')
compare_methods(threshold_map, rf_map)

In [ ]:
fig = plot_method_comparison(threshold_map, rf_map, proba_map)
plt.show()

---
## 6. Feature Importance

In [ ]:
importance_df = rf_clf.feature_importance_df()
print('Top 10 features:')
print(importance_df.head(10).to_string(index=False))

fig = plot_feature_importance(importance_df, top_n=15)
plt.show()

---
## 7. Interactive Folium Map

In [ ]:
m = build_folium_map(
    tif_path=STACK_PATH,
    threshold_map=threshold_map,
    rf_map=rf_map,
    proba_map=proba_map,
    out_path='../outputs/maps/burn_map_interactive.html'
)
m  # renders inline in Jupyter

---
## 8. Permutation Importance (Robust Feature Analysis)

MDI (Gini impurity) overestimates importance of correlated features.
Permutation importance gives a more honest view.

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

perm = permutation_importance(
    rf_clf.model, X_te, y_te,
    n_repeats=10, random_state=42, n_jobs=-1,
    scoring='f1'
)

perm_df = pd.DataFrame({
    'feature':    feature_cols,
    'importance': perm.importances_mean,
    'std':        perm.importances_std
}).sort_values('importance', ascending=False)

print('Permutation importance (top 10):')
print(perm_df.head(10).to_string(index=False))